### Multi-Disease GEO Data Collection

Download and inspect raw gene expression datasets from NCBI GEO for four autoimmune diseases:
- Vitiligo (GSE65127)
- Systemic Lupus Erythematosus (GSE318067)
- Rheumatoid Arthritis (GSE93272)
- Type 1 Diabetes (GSE55098)

Key genes of interest: PTPN22, NLRP1

In [ ]:
import GEOparse
import pandas as pd
import os

# Set data path relative to project root
os.makedirs("../data/raw", exist_ok=True)

print("Libraries loaded successfully")
print(f"GEOparse version: {GEOparse.__version__}")

In [ ]:
data_raw = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/raw")
os.makedirs(data_raw, exist_ok=True)

gse65127 = GEOparse.get_GEO(geo="GSE65127", destdir=data_raw, silent=False, how="full")

In [ ]:
# Extract expression matrix from all samples
expr_matrix = gse65127.pivot_samples("VALUE")

print(expr_matrix.shape)
print(expr_matrix.head())

In [ ]:
# Get the annotation table from GPL570
gpl = gse65127.gpls["GPL570"]
annotation = gpl.table

print(annotation.shape)
print(annotation.columns.tolist())
print(annotation.head())

In [ ]:
# Keep only ID and Gene Symbol columns
gene_map = annotation[["ID", "Gene Symbol"]].copy()
gene_map = gene_map.dropna(subset=["Gene Symbol"])
gene_map = gene_map[gene_map["Gene Symbol"] != ""]

print(gene_map.shape)
print(gene_map.head())

In [ ]:
# Keep only unambiguous probes (no /// in gene symbol)
gene_map_clean = gene_map[~gene_map["Gene Symbol"].str.contains("///")]

print(f"Before cleaning: {gene_map.shape[0]} probes")
print(f"After cleaning: {gene_map_clean.shape[0]} probes")
print(gene_map_clean.head())

In [ ]:
# Merge expression matrix with gene annotation
expr_matrix_reset = expr_matrix.reset_index()

merged = expr_matrix_reset.merge(gene_map_clean, left_on="ID_REF", right_on="ID", how="inner")

# Set gene symbol as index
merged = merged.drop(columns=["ID_REF", "ID"])
merged = merged.set_index("Gene Symbol")

print(merged.shape)
print(merged.head())

In [ ]:
# Average expression values for duplicate genes
merged_clean = merged.groupby("Gene Symbol").mean()

print(f"Before deduplication: {merged.shape[0]} genes")
print(f"After deduplication: {merged_clean.shape[0]} genes")
print(merged_clean.head())

In [ ]:
# Verify no duplicates remain
print(merged_clean.index.duplicated().sum())

In [ ]:
# Save processed GSE65127 expression matrix
data_processed = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/processed")
os.makedirs(data_processed, exist_ok=True)

merged_clean.to_csv(f"{data_processed}/GSE65127_vitiligo_expr.csv")
print(f"Saved: GSE65127_vitiligo_expr.csv")
print(f"Shape: {merged_clean.shape}")

In [ ]:
def process_geo_dataset(geo_id, disease_name, data_raw, data_processed):
    print(f"\nProcessing {geo_id} — {disease_name}")
    
    # Download with fallback
    try:
        gse = GEOparse.get_GEO(geo=geo_id, destdir=data_raw, silent=True, how="full")
    except Exception as e:
        print(f"First attempt failed: {e}")
        print("Retrying...")
        gse = GEOparse.get_GEO(geo=geo_id, destdir=data_raw, silent=False, how="full")
    
    # Expression matrix
    expr = gse.pivot_samples("VALUE")
    
    # Get annotation
    gpl_id = list(gse.gpls.keys())[0]
    annotation = gse.gpls[gpl_id].table
    gene_map = annotation[["ID", "Gene Symbol"]].copy()
    gene_map = gene_map.dropna(subset=["Gene Symbol"])
    gene_map = gene_map[gene_map["Gene Symbol"] != ""]
    gene_map = gene_map[~gene_map["Gene Symbol"].str.contains("///")]
    
    # Merge and clean
    expr_reset = expr.reset_index()
    merged = expr_reset.merge(gene_map, left_on="ID_REF", right_on="ID", how="inner")
    merged = merged.drop(columns=["ID_REF", "ID"])
    merged = merged.set_index("Gene Symbol")
    merged = merged.groupby("Gene Symbol").mean()
    
    # Save
    filename = f"{geo_id}_{disease_name}_expr.csv"
    merged.to_csv(f"{data_processed}/{filename}")
    print(f"Saved: {filename} — Shape: {merged.shape}")
    
    return merged

In [ ]:
## GSE93272 and GSE55098 downloaded manually via curl due to GEOparse FTP instability
## Files saved to data/raw/ then loaded from disk by GEOparse

# datasets = [
#    ("GSE318067", "SLE"),
#    ("GSE93272", "RA"),
#    ("GSE55098", "T1D")
#]

results = {}
for geo_id, disease_name in datasets:
    results[geo_id] = process_geo_dataset(geo_id, disease_name, data_raw, data_processed)